<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/710_opencv_%EC%98%81%EC%83%81_Canny_%2B_Hough_%EC%B0%A8%EC%84%A0_%EA%B0%90%EC%A7%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: OpenCV로 Canny + Hough로 차선 감지

Step 2: ROI 적용 및 영상 처리

Step 3: CNN을 사용한 간단한 edge 분류 (후속 실습)

In [ ]:
# 필요한 라이브러리 임포트
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from google.colab.patches import cv2_imshow
import os

# OpenCV 버전 확인
print(f"OpenCV 버전: {cv2.__version__}")

# 영상 파일 경로 설정 (예시)
video_path = '/content/김영빈_영상.mp4'

In [ ]:
# yt-dlp 설치
!pip install yt-dlp

# 원하는 유튜브 영상 다운로드
!yt-dlp -f bestvideo+bestaudio --merge-output-format mp4 https://www.youtube.com/watch?v=tEtWnGwwCEc


유튜브 영상 다운로드 한 후 인식

In [ ]:
import cv2
import yt_dlp
import matplotlib.pyplot as plt
import tempfile
import os
import time
from IPython.display import clear_output

def play_youtube_video(youtube_url, skip_frames=1):
    """
    YouTube 영상을 다운로드하고 재생하는 간단한 함수
    skip_frames: 프레임 건너뛰기 (1=모든 프레임, 2=한 프레임씩 건너뛰기)
    """

    ydl_opts = {
        'format': 'mp4/best[height<=480]',  # 480p로 제한 (빠른 처리)
        'outtmpl': tempfile.gettempdir() + '/temp_video.%(ext)s',
        'quiet': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            print("⬇️ 영상 다운로드 중...")
            info = ydl.extract_info(youtube_url, download=True)

            video_path = ydl.prepare_filename(info)

            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            print(f"🎬 '{info['title']}' 재생 시작!")
            print("(Ctrl+C로 중단)")

            frame_num = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                # 프레임 건너뛰기
                if frame_num % skip_frames == 0:
                    clear_output(wait=True)
                    plt.figure(figsize=(10, 6))
                    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                    plt.title(f"재생 중... ({frame_num}/{frame_count})")
                    plt.axis('off')
                    plt.show()

                    # 재생 속도 조절
                    time.sleep(0.1)  # 0.1초 대기 (빠른 재생)

                frame_num += 1

            cap.release()
            os.remove(video_path)
            print("✅ 재생 완료!")

    except KeyboardInterrupt:
        print("\n⏹️ 재생 중단")
    except Exception as e:
        print(f"❌ 오류: {e}")

# 사용법
youtube_url = input("YouTube URL: ")
play_youtube_video(youtube_url, skip_frames=2)  # 한 프레임씩 건너뛰어 빠른 재생

미션  이 코드에서 1. 영상 저장 설정 2. 2. 프레임 저장  3. output 리소스 정리 4. 4. 결과 영상을 /content/lane_detection_result.mp4에 다운로드하게 만들어 주세요

In [ ]:
import cv2
from google.colab import files
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
import time

# 영상 업로드
#uploaded = files.upload()
video_path = "/content/sample_data/around5.mp4"

# 영상 열기
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"📹 영상 정보: {frame_count}프레임, {fps:.2f}FPS")
print("🎬 차선 인식 시작! (Ctrl+C로 중단)")

frame_num = 0
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("📺 영상 재생 완료")
            break

        # 1. 그레이스케일 변환
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 2. 블러 → 에지(Canny) - 임계값 낮춤
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 30, 100)  # 50,150 → 30,100

        # 3. ROI 설정 - 하단 30%
        height, width = edges.shape
        mask = np.zeros_like(edges)
        polygon = np.array([[
            (int(width * 0.1), height),            # 왼쪽 아래
            (int(width * 0.9), height),            # 오른쪽 아래
            (int(width * 0.6), int(height * 0.7)), # 오른쪽 위 (70% 지점)
            (int(width * 0.4), int(height * 0.7))  # 왼쪽 위 (70% 지점)
        ]])
        cv2.fillPoly(mask, polygon, 255)
        roi = cv2.bitwise_and(edges, mask)

        # 4. Hough Transform으로 직선 검출 - 파라미터 완화
        lines = cv2.HoughLinesP(roi, 2, np.pi / 180,
                               threshold=30,      # 50 → 30
                               minLineLength=30,  # 40 → 30
                               maxLineGap=80)     # 50 → 80

        # 5. 원본 프레임에 선 그리기
        line_image = frame.copy()
        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 5)

        # 6. 결과 출력
        clear_output(wait=True)
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.title(f"원본 프레임 {frame_num}")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(cv2.cvtColor(line_image, cv2.COLOR_BGR2RGB))
        plt.title(f"차선 인식 결과 ({len(lines) if lines is not None else 0}개)")
        plt.axis('off')
        plt.show()

        frame_num += 1
        time.sleep(10)  # 10초마다 이걸 바꿔본다.

except KeyboardInterrupt:
    print("\n⏹️ 재생 중단")

cap.release()

미션 결과 코드

In [ ]:
import cv2
from google.colab import files
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
import time

# 영상 업로드
#uploaded = files.upload()
video_path = "/content/sample_data/around5.mp4"

# 영상 열기
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# ROI 설정 (도로 영역만 분석) - 좌표는 영상에 맞게 조정 필요
def create_roi_mask(frame):
    height, width = frame.shape[:2]
    mask = np.zeros((height, width), dtype=np.uint8)
    # 사다리꼴 모양의 도로 영역 설정
    roi_points = np.array([
        [int(width * 0.1), height],                    # 왼쪽 아래
        [int(width * 0.9), height],                    # 오른쪽 아래
        [int(width * 0.6), int(height * 0.7)],         # 오른쪽 위 (70% 지점)
        [int(width * 0.4), int(height * 0.7)]          # 왼쪽 위 (70% 지점)
    ], np.int32)
    cv2.fillPoly(mask, [roi_points], 255)
    return mask

print(f"📹 영상 정보: {frame_count}프레임, {fps:.2f}FPS, {width}x{height}")
print("🎬 차선 인식 시작! (Ctrl+C로 중단)")

# 결과 영상 저장 설정
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('/content/lane_detection_result.mp4', fourcc, fps, (width, height))

frame_num = 0
display_interval = int(fps * 10)  # 10초마다 (fps * 10 프레임마다)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("📺 영상 재생 완료")
            break

        # 1. 그레이스케일 변환
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 2. 블러 → 에지(Canny) - 임계값 낮춤
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 30, 100)  # 50,150 → 30,100

        # 3. ROI 설정 - 도로 영역만 분석
        mask = create_roi_mask(edges)
        roi = cv2.bitwise_and(edges, mask)

        # 4. Hough Transform으로 직선 검출 - 파라미터 완화
        lines = cv2.HoughLinesP(roi, 2, np.pi / 180,
                               threshold=30,      # 50 → 30
                               minLineLength=30,  # 40 → 30
                               maxLineGap=80)     # 50 → 80

        # 5. 원본 프레임에 선 그리기
        line_image = frame.copy()

        # ROI 영역을 화면에 표시 (사다리꼴 테두리)
        roi_points = np.array([
            [int(width * 0.1), height],                    # 왼쪽 아래
            [int(width * 0.9), height],                    # 오른쪽 아래
            [int(width * 0.6), int(height * 0.7)],         # 오른쪽 위 (70% 지점)
            [int(width * 0.4), int(height * 0.7)]          # 왼쪽 위 (70% 지점)
        ], np.int32)
        cv2.polylines(line_image, [roi_points], isClosed=True, color=(255, 0, 0), thickness=2)

        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 5)

        # ★ 결과 영상에 프레임 저장 ★
        out.write(line_image)

        # 6. 10초마다 1프레임씩 결과 보여주기
        if frame_num % display_interval == 0:
            # 원본과 결과를 나란히 배치
            combined = np.hstack((frame, line_image))

            clear_output(wait=True)
            plt.figure(figsize=(15, 6))
            plt.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
            plt.title(f"프레임 {frame_num} - 원본 | 차선 인식 결과 ({len(lines) if lines is not None else 0}개)")
            plt.axis('off')
            plt.show()

        # 진행률 표시 (매 프레임마다)
        print(f"\r📊 진행률: {frame_num+1}/{frame_count} ({(frame_num+1)/frame_count*100:.1f}%)", end='')

        frame_num += 1

except KeyboardInterrupt:
    print("\n⏹️ 재생 중단 - 결과 저장 중...")

# 리소스 정리
cap.release()
out.release()

print("\n✅ 처리 완료!")
print("📹 결과 영상이 저장되었습니다: /content/lane_detection_result.mp4")

# 결과 영상 다운로드
print("📥 결과 영상 다운로드 중...")
files.download('/content/lane_detection_result.mp4')

조금 개선된 코드  미션  roi 사다리꼴 출력되게 만드세요

In [ ]:
import cv2
from google.colab import files
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
import time

# 바탕화면에서 영상 파일 업로드
print("📤 바탕화면에 있는 영상 파일을 업로드하세요!")
uploaded = files.upload()

# 업로드된 파일 이름 가져오기
video_filename = list(uploaded.keys())[0]
video_path = f"/content/{video_filename}"

print(f"✅ 업로드 완료: {video_filename}")

# 영상 열기
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"📹 영상 정보: {frame_count}프레임, {fps:.2f}FPS, {width}x{height}")
print("🎬 차선 인식 시작! (Ctrl+C로 중단)")

# 결과 영상 저장 설정
result_filename = f"lane_detection_result_{video_filename}"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(f'/content/{result_filename}', fourcc, fps, (width, height))

frame_num = 0
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("📺 영상 재생 완료")
            break

        # 1. 그레이스케일 변환
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 2. 블러 → 에지(Canny) - 임계값 더 낮춤 (인식률 향상)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 20, 80)  # 30,100 → 20,80 (더 민감하게)

        # 3. ROI 설정 - 더 넓은 영역으로 확장
        img_height, img_width = edges.shape
        mask = np.zeros_like(edges)
        polygon = np.array([[
            (int(img_width * 0.1), img_height),           # 왼쪽 아래 (더 넓게: 0.1→0.05)
            (int(img_width * 0.9), img_height),           # 오른쪽 아래 (더 넓게: 0.9→0.95)
            (int(img_width * 0.6), int(img_height * 0.7)), # 오른쪽 위 (더 높게: 0.7→0.6)
            (int(img_width * 0.4), int(img_height * 0.7))  # 왼쪽 위 (더 높게: 0.7→0.6)

        ]])
        cv2.fillPoly(mask, polygon, 255)
        roi = cv2.bitwise_and(edges, mask)

        # 4. Hough Transform으로 직선 검출 - 파라미터 대폭 완화 (인식률 향상)
        lines = cv2.HoughLinesP(roi, 1, np.pi / 180,      # rho: 2→1 (더 정밀)
                               threshold=20,      # 30 → 20 (더 관대하게)
                               minLineLength=20,  # 30 → 20 (짧은 선도 인식)
                               maxLineGap=100)    # 80 → 100 (끊어진 선 연결)

        # 5. 원본 프레임에 선 그리기
        line_image = frame.copy()
        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 5)

        # ★ 결과 영상에 프레임 저장 ★
        out.write(line_image)

        # 6. 결과 출력 (번쩍거림 방지)
        # 매 10프레임마다만 화면 출력 (또는 원하는 간격으로 조정)
        if frame_num % 10 == 0:  # 10프레임마다 출력
            clear_output(wait=True)
            plt.figure(figsize=(12, 6))
            plt.subplot(1, 2, 1)
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title(f"원본 프레임 {frame_num}")
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(cv2.cvtColor(line_image, cv2.COLOR_BGR2RGB))
            plt.title(f"차선 인식 결과 ({len(lines) if lines is not None else 0}개)")
            plt.axis('off')
            plt.show()

        # 진행률 표시 (텍스트만, 번쩍거리지 않음)
        if frame_num % 30 == 0:  # 30프레임마다 진행률 출력
            print(f"📊 진행률: {frame_num+1}/{frame_count} ({(frame_num+1)/frame_count*100:.1f}%)")

        frame_num += 1
        time.sleep(1)  # 1초로 변경

except KeyboardInterrupt:
    print("\n⏹️ 재생 중단 - 결과 저장 중...")

# 리소스 정리
cap.release()
out.release()

print("\n✅ 처리 완료!")
print(f"📹 결과 영상이 저장되었습니다: /content/{result_filename}")

# 결과 영상 다운로드
print("📥 결과 영상 다운로드 중...")
files.download(f'/content/{result_filename}')

In [ ]:
import cv2
from google.colab import files
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
import time

# 바탕화면에서 영상 파일 업로드
print("📤 바탕화면에 있는 영상 파일을 업로드하세요!")
#uploaded = files.upload()

# 업로드된 파일 이름 가져오기 (경로 수정)
#video_filename = list(uploaded.keys())[0]
video_path = "/content/sample_data/around5.mp4"  # 전체 경로
video_filename = "around5.mp4"  # 파일명만
print(f"✅ 파일 경로: {video_path}")

# 영상 열기
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"📹 영상 정보: {frame_count}프레임, {fps:.2f}FPS, {width}x{height}")
print("🎬 차선 인식 시작! (Ctrl+C로 중단)")

# 결과 영상 저장 설정 (파일명 수정)
result_filename = f"lane_detection_result_{video_filename}"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(f'/content/{result_filename}', fourcc, fps, (width, height))

frame_num = 0
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("📺 영상 재생 완료")
            break

        # 1. 그레이스케일 변환
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 2. 히스토그램 평활화 (대비 개선)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(gray)

        # 3. 블러 → 에지(Canny) - 차선에 맞게 조정
        blur = cv2.GaussianBlur(enhanced, (5, 5), 0)  # 더 선명하게
        edges = cv2.Canny(blur, 30, 100)  # 더 민감하게 (차선 검출용)

        # 4. ROI 설정 - 대시보드 완전 제외, 도로만 집중
        img_height, img_width = edges.shape
        mask = np.zeros_like(edges)
        polygon = np.array([[
            (int(img_width * 0.2), img_height),            # 왼쪽 아래
            (int(img_width * 0.9), img_height),            # 오른쪽 아래
            (int(img_width * 0.7), int(img_height * 0.65)), # 오른쪽 위
            (int(img_width * 0.4), int(img_height * 0.65))  # 왼쪽 위
        ]])
        cv2.fillPoly(mask, polygon, 255)
        roi = cv2.bitwise_and(edges, mask)

        # 5. Hough Transform - 차선 검출에 특화
        lines = cv2.HoughLinesP(roi, 1, np.pi / 180,
                               threshold=30,      # 낮춤 (더 많은 선 검출)
                               minLineLength=50,  # 적당한 길이
                               maxLineGap=100)    # 끊어진 차선도 연결

        # 6. 차선 필터링 및 그리기
        line_image = frame.copy()
        lane_count = 0

        if lines is not None:
            # 각도와 길이로 차선 필터링
            filtered_lines = []
            for line in lines:
                x1, y1, x2, y2 = line[0]

                # 선의 길이 계산
                length = np.sqrt((x2-x1)**2 + (y2-y1)**2)

                # 선의 각도 계산
                if x2 - x1 != 0:
                    angle = np.arctan((y2 - y1) / (x2 - x1)) * 180 / np.pi

                    # 차선 조건: 적당한 각도 (20도~80도) + 충분한 길이 (30픽셀 이상)
                    if (20 <= abs(angle) <= 80) and length >= 30:
                        filtered_lines.append(line)

            lane_count = len(filtered_lines)

            # 필터링된 차선만 그리기
            for line in filtered_lines:
                x1, y1, x2, y2 = line[0]
                cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 5)

        # ★ 결과 영상에 프레임 저장 ★
        out.write(line_image)

        # 7. 결과 출력 (번쩍거림 방지)
        if frame_num % 10 == 0:  # 10프레임마다 출력
            clear_output(wait=True)
            plt.figure(figsize=(20, 12))

            # ROI 경계선을 원본에 표시
            frame_with_roi = frame.copy()
            cv2.polylines(frame_with_roi, [polygon], True, (255, 0, 0), 3)  # 파란색 ROI 경계선

            # 원본
            plt.subplot(2, 4, 1)
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title(f"원본 프레임 {frame_num}")
            plt.axis('off')

            # ROI 경계선 표시
            plt.subplot(2, 4, 2)
            plt.imshow(cv2.cvtColor(frame_with_roi, cv2.COLOR_BGR2RGB))
            plt.title("ROI 경계선 (파란색)")
            plt.axis('off')

            # 에지 검출 (전체)
            plt.subplot(2, 4, 3)
            plt.imshow(edges, cmap='gray')
            plt.title("전체 에지 검출")
            plt.axis('off')

            # ROI 마스크
            plt.subplot(2, 4, 4)
            plt.imshow(mask, cmap='gray')
            plt.title("ROI 마스크")
            plt.axis('off')

            # ROI 적용된 에지
            plt.subplot(2, 4, 5)
            plt.imshow(roi, cmap='gray')
            plt.title("ROI 적용된 에지")
            plt.axis('off')

            # 검출된 직선들만
            lines_only = np.zeros_like(frame)
            if lines is not None:
                for line in lines:
                    x1, y1, x2, y2 = line[0]
                    cv2.line(lines_only, (x1, y1), (x2, y2), (0, 255, 0), 3)

            plt.subplot(2, 4, 6)
            plt.imshow(cv2.cvtColor(lines_only, cv2.COLOR_BGR2RGB))
            plt.title("검출된 모든 직선")
            plt.axis('off')

            # 필터링된 차선만
            filtered_only = np.zeros_like(frame)
            if lines is not None:
                for line in lines:
                    x1, y1, x2, y2 = line[0]
                    length = np.sqrt((x2-x1)**2 + (y2-y1)**2)
                    if x2 - x1 != 0:
                        angle = np.arctan((y2 - y1) / (x2 - x1)) * 180 / np.pi
                        if (20 <= abs(angle) <= 80) and length >= 30:
                            cv2.line(filtered_only, (x1, y1), (x2, y2), (0, 255, 0), 3)

            plt.subplot(2, 4, 7)
            plt.imshow(cv2.cvtColor(filtered_only, cv2.COLOR_BGR2RGB))
            plt.title("필터링된 차선")
            plt.axis('off')

            # 최종 결과
            plt.subplot(2, 4, 8)
            plt.imshow(cv2.cvtColor(line_image, cv2.COLOR_BGR2RGB))
            plt.title(f"최종 결과 ({lane_count}개)")
            plt.axis('off')

            plt.tight_layout()
            plt.show()

            # ROI 좌표 정보 출력 (실제 사용된 좌표로 수정)
            print(f"🔍 ROI 좌표 정보:")
            print(f"   왼쪽 아래: ({int(img_width * 0.2)}, {img_height})")
            print(f"   오른쪽 아래: ({int(img_width * 0.8)}, {img_height})")
            print(f"   오른쪽 위: ({int(img_width * 0.6)}, {int(img_height * 0.65)})")
            print(f"   왼쪽 위: ({int(img_width * 0.4)}, {int(img_height * 0.65)})")

        # 진행률 표시
        if frame_num % 30 == 0:  # 30프레임마다 진행률 출력
            progress = (frame_num + 1) / frame_count * 100
            print(f"📊 진행률: {frame_num+1}/{frame_count} ({progress:.1f}%)")

        frame_num += 1
        time.sleep(0.1)  # 1초 → 0.1초로 수정 (더 빠른 업데이트)

except KeyboardInterrupt:
    print("\n⏹️ 재생 중단 - 결과 저장 중...")

# 리소스 정리
cap.release()
out.release()

print("\n✅ 처리 완료!")
print(f"📹 결과 영상이 저장되었습니다: /content/{result_filename}")

# 결과 영상 다운로드
print("📥 결과 영상 다운로드 중...")
try:
    files.download(f'/content/{result_filename}')
    print("✅ 다운로드 완료!")
except Exception as e:
    print(f"❌ 다운로드 실패: {e}")
    print("파일이 존재하는지 확인하세요.")

너무 인식이 안되서 다시 roi를 보기로한다.

In [ ]:
import cv2
from google.colab import files
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
import time

# 바탕화면에서 영상 파일 업로드
print("📤 바탕화면에 있는 영상 파일을 업로드하세요!")
#uploaded = files.upload()

# 업로드된 파일 이름 가져오기
#video_filename = list(uploaded.keys())[0]
video_path = "/content/sample_data/around5.mp4"
video_filename = "around5.mp4"  # 직접 파일명 지정

print(f"✅ 파일 경로: {video_path}")

# 영상 열기
cap = cv2.VideoCapture(video_path)

# 영상 파일이 제대로 열렸는지 확인
if not cap.isOpened():
    print("❌ 영상 파일을 열 수 없습니다. 경로를 확인해주세요.")
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"📹 영상 정보: {frame_count}프레임, {fps:.2f}FPS, {width}x{height}")
print("🎬 차선 인식 시작! (Ctrl+C로 중단)")

# 결과 영상 저장 설정
result_filename = f"lane_detection_result_{video_filename}"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(f'/content/{result_filename}', fourcc, fps, (width, height))

frame_num = 0
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("📺 영상 재생 완료")
            break

        # 1. 그레이스케일 변환
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 2. 블러 → 에지(Canny) - 적당한 민감도로 조정
        blur = cv2.GaussianBlur(gray, (7, 7), 0)  # 더 부드럽게
        edges = cv2.Canny(blur, 40, 120)  # 덜 민감하게

        # 3. ROI 설정 - 차선 영역에 더 집중
        img_height, img_width = edges.shape
        mask = np.zeros_like(edges)
        polygon = np.array([[
            (int(img_width * 0.2), img_height),            # 왼쪽 아래
            (int(img_width * 0.8), img_height),            # 오른쪽 아래
            (int(img_width * 0.6), int(img_height * 0.65)), # 오른쪽 위
            (int(img_width * 0.4), int(img_height * 0.65))  # 왼쪽 위
        ]])
        cv2.fillPoly(mask, polygon, 255)
        roi = cv2.bitwise_and(edges, mask)

        # 4. Hough Transform으로 직선 검출
        lines = cv2.HoughLinesP(roi, 2, np.pi / 180,
                               threshold=40,      # 더 엄격하게
                               minLineLength=40,  # 긴 선만 인식
                               maxLineGap=60)     # 연결 기준 강화

        # 5. 원본 프레임에 선 그리기
        line_image = frame.copy()
        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 5)

        # ★ 결과 영상에 프레임 저장 ★
        out.write(line_image)

        # 6. 결과 출력 (번쩍거림 방지)
        if frame_num % 10 == 0:  # 10프레임마다 출력
            clear_output(wait=True)
            plt.figure(figsize=(12, 6))
            plt.subplot(1, 2, 1)
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title(f"원본 프레임 {frame_num}")
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(cv2.cvtColor(line_image, cv2.COLOR_BGR2RGB))
            plt.title(f"차선 인식 결과 ({len(lines) if lines is not None else 0}개)")
            plt.axis('off')
            plt.show()

        # 진행률 표시
        if frame_num % 10 == 0:  # 30프레임마다 진행률 출력
            progress = (frame_num + 1) / frame_count * 100
            print(f"📊 진행률: {frame_num+1}/{frame_count} ({progress:.1f}%)")

        frame_num += 1
        time.sleep(0.1)  # 10 → 0.1로 수정 (10초는 너무 김)

except KeyboardInterrupt:
    print("\n⏹️ 재생 중단 - 결과 저장 중...")

# 리소스 정리
cap.release()
out.release()

print("\n✅ 처리 완료!")
print(f"📹 결과 영상이 저장되었습니다: /content/{result_filename}")

# 결과 영상 다운로드
print("📥 결과 영상 다운로드 중...")
files.download(f'/content/{result_filename}')

polygon = np.array([[
    # 사다리꼴 모양:      /─────\  ← 30%~70% (넓어진 상단)
    # 아래쪽으로 갈수록: /───────\  
    # 점점 넓어져서:    /_________\ ← 0%~100% (전체 하단)
   
]])

            (int(img_width * 0.2), img_height),            # 왼쪽 아래
            (int(img_width * 0.8), img_height*0.6),            # 오른쪽 아래
            (int(img_width * 0.6), int(img_height * 0.65)), # 오른쪽 위
            (int(img_width * 0.4), int(img_height * 0.65))  # 왼쪽 위